# Serving Ministral 3 8B on Kaggle and calling it from VS Code

This is the setup we used to benchmark a small open-weight vision model on real software-development tasks (SWE-bench Multimodal). The model runs on a free Kaggle notebook with two Tesla T4 GPUs. We serve it with vLLM, expose it to the internet with a cloudflared tunnel, and call it from a laptop with the standard `openai` Python library, since vLLM speaks the OpenAI API format.

The agent that actually did the benchmark ran on a Mac, because Kaggle has no Docker and the test evaluation needs it. Kaggle only hosts the model.

This is a reference guide for the setup we used to benchmark a small open-weight vision model on real software-development tasks (SWE-bench Multimodal). The model runs on a free Kaggle notebook, and everything else runs on a laptop.

The notebook covers:

1. Checking the machine and installing vLLM
2. Starting the server and testing it
3. Exposing it with a tunnel
4. Loading the SWE-bench Multimodal data and sending a real issue screenshot
5. Recording GPU usage and running a load test
6. Calling the endpoint from VS Code
7. Troubleshooting

Sections 1-10 run on Kaggle with the accelerator set to GPU T4 x2 and Internet on. Section 11 runs on your own computer. The agent that did the benchmark ran on a Mac, because Kaggle has no Docker and the test evaluation needs it. Kaggle only hosts the model.

## 1. Check the machine
`nvidia-smi` lists the GPUs (the `!` runs the line as a terminal command). We expect two Tesla T4s with about 15 GB each and almost nothing in use. The second cell asks PyTorch the same question and also prints the compute capability, which is 7.5 for a T4. That number matters later: it is why we can't use BF16, FP8 or FlashAttention 2 on this hardware.

Two Kaggle folders to know about: `/kaggle/temp` is fast but wiped when the session ends, so the Python environment and the model cache live there. `/kaggle/working` usually survives a reset, so logs and CSV files go there.

In [ ]:
!nvidia-smi

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)
    print(f"GPU {i}:", gpu.name)
    print("  VRAM:", round(gpu.total_memory / 1024**3, 2), "GB")
    print("  Compute capability:", f"{gpu.major}.{gpu.minor}")

## 2. Check Internet and install vLLM
The install downloads from PyPI and the model comes from Hugging Face, so first confirm both are reachable. Each line should print `HTTP/2 200`. If it can't resolve the host, Internet is switched off in the Settings panel.

vLLM goes into its own virtual environment under `/kaggle/temp`. It pins exact versions of PyTorch and the CUDA libraries, and we don't want that to clash with Kaggle's own packages. `uv` is a fast installer, and `--torch-backend=auto` picks the PyTorch build that matches the GPU driver. This takes roughly 5-8 minutes, and you can ignore the red dependency warnings about other Kaggle packages.

Because `/kaggle/temp` is wiped when a session ends, we ran this cell again every time Kaggle reset (three times during the benchmark). The versions we ended up with were vLLM 0.34.0 and PyTorch 2.14.0 (CUDA 13.0 build on the first run, 13.2 after a reset).

In [ ]:
!curl -sI https://pypi.org | head -1
!curl -sI https://huggingface.co | head -1

In [ ]:
!pip install -q uv
!rm -rf /kaggle/temp/venv
!uv venv /kaggle/temp/venv --python 3.12
!uv pip install --python /kaggle/temp/venv/bin/python vllm --torch-backend=auto
!/kaggle/temp/venv/bin/python -c "import torch, vllm; print('vLLM', vllm.__version__, '| torch', torch.__version__, '| cuda ok:', torch.cuda.is_available(), torch.cuda.device_count())"

## 3. Start the server
`vllm serve` is the whole deployment. It downloads the model, splits it across both GPUs and starts an API on port 8000. We run it with `subprocess.Popen` so it stays in the background and the notebook remains usable. Its output goes to a log file, and a random API key is generated on every launch.

The model is `mistralai/Ministral-3-8B-Instruct-2512-BF16`. The official release is FP8, which a T4 can't run, so we use the BF16 copy and let vLLM convert it to FP16 on load.

What the options do:

- `--dtype half` runs in FP16, because T4s have no BF16.
- `--tensor-parallel-size 2` splits each layer across both GPUs, about 8.4 GiB of weights on each.
- `--max-model-len 32768` is the longest conversation (prompt plus answer) the server accepts. We started at 16K and raised it to 32K because coding prompts with screenshots get long.
- `--gpu-memory-utilization 0.90` lets vLLM claim 90% of each GPU. What the weights don't use becomes the KV cache, the memory the model uses to remember the conversation so far.
- `--tokenizer-mode hf --config-format hf --load-format safetensors` loads the model from its Hugging Face files. This is required for image input. See Troubleshooting at the end for the error you get without it.
- `--enable-auto-tool-choice --tool-call-parser mistral` turns on tool calling. The parser has to match the model's own tool-call format.
- `--limit-mm-per-prompt '{"image": 4}'` allows up to 4 images per request.
- `--api-key` makes the server reject any `/v1` request that doesn't carry the key.

The loop checks `/health` every 10 seconds and prints how long startup took. The first launch needs the download, so allow 5-12 minutes. With the files already cached it was about 3 minutes.

In [ ]:
import os, subprocess, time, secrets, urllib.request

API_KEY = secrets.token_urlsafe(24)
env = os.environ.copy(); env["HF_HOME"] = "/kaggle/temp/hf"

cmd = ["/kaggle/temp/venv/bin/vllm", "serve", "mistralai/Ministral-3-8B-Instruct-2512-BF16",
       "--served-model-name", "ministral-3-8b", "--dtype", "half",
       "--tokenizer-mode", "hf", "--config-format", "hf", "--load-format", "safetensors",
       "--tensor-parallel-size", "2", "--max-model-len", "32768", "--gpu-memory-utilization", "0.90",
       "--enable-auto-tool-choice", "--tool-call-parser", "mistral",
       "--limit-mm-per-prompt", '{"image": 4}',
       "--port", "8000", "--api-key", API_KEY]
log = open("/kaggle/working/vllm_ministral.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)

t0 = time.time()
while time.time() - t0 < 1500:
    if server.poll() is not None:
        print("Server exited early, code", server.returncode); break
    try:
        if urllib.request.urlopen("http://localhost:8000/health", timeout=5).status == 200:
            print(f"Ready after {int(time.time()-t0)} s"); break
    except Exception:
        pass
    time.sleep(10)

## 4. Read the startup log
The log shows how the GPU memory was used. The lines we care about: which model class vLLM chose (it should say `Mistral3ForConditionalGeneration` in HF mode), which attention backend it uses (`TRITON_ATTN` on T4), how much memory the weights took per GPU, how many tokens fit in the KV cache, and how many full-length requests can run at once.

In our run the weights took about 8.43 GiB per GPU, the KV cache held 48,128 tokens, and concurrency at 32K context was about 1.47. That last number is the real limit on how many users the server can handle: at full length it fits barely more than one conversation. If the server fails to start, search the same log for `Error` or `Traceback`.

In [ ]:
!grep -n -iE "attention backend|KV cache size|Maximum concurrency|Model loading took|Resolved architecture" /kaggle/working/vllm_ministral.log | cut -c1-230 | head -12

## 5. Test the server from inside Kaggle
Before exposing anything, check that the server works locally. The client is the normal `openai` library, pointed at `localhost` with our key. We test four things:

1. A tool call. We describe a function `read_file` and ask the model to open a file. A working setup returns a structured tool call instead of text. Coding agents are built on this.
2. An image by URL.
3. Two images at once as base64 data URLs, which is how a program sends local screenshots. This is how the benchmark sent issue screenshots.
4. A long prompt of about 20K tokens, to confirm the 32K window really works.

Each image adds roughly 1-3K tokens to `prompt_tokens`, so you can see what images cost. In our run the 2-image request used about 5.3K prompt tokens and the long-context test about 18.5K.

In [ ]:
import base64, urllib.request
from openai import OpenAI

c = OpenAI(base_url="http://localhost:8000/v1", api_key=API_KEY)
M = "ministral-3-8b"

# 1) Tool call
tools = [{"type": "function", "function": {
    "name": "read_file", "description": "Read a file from the repository",
    "parameters": {"type": "object",
                   "properties": {"path": {"type": "string", "description": "File path"}},
                   "required": ["path"]}}}]
r = c.chat.completions.create(model=M, temperature=0.0, max_tokens=150, tools=tools, tool_choice="auto",
    messages=[{"role": "user", "content": "Open src/utils.py and show me its contents."}])
print("TOOL CALLS:", r.choices[0].message.tool_calls, "\n")

def ask(content, label):
    try:
        r = c.chat.completions.create(model=M, temperature=0.2, max_tokens=80,
            messages=[{"role": "user", "content": content}])
        print(label, r.choices[0].message.content, "| prompt_tokens:", r.usage.prompt_tokens, "\n")
    except Exception as e:
        print(label, "FAILED:", str(e)[:250], "\n")

# 2) Image via URL
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
ask([{"type": "image_url", "image_url": {"url": url}},
     {"type": "text", "text": "What animal is on the candy?"}], "IMAGE (url):")

# 3) Two images via base64
b64 = base64.b64encode(urllib.request.urlopen(url).read()).decode()
d = f"data:image/jpeg;base64,{b64}"
ask([{"type": "image_url", "image_url": {"url": d}},
     {"type": "image_url", "image_url": {"url": d}},
     {"type": "text", "text": "How many images do you see, and what is in them?"}], "IMAGE (2x base64):")

# 4) Long-context check
filler = "The quick brown fox jumps over the lazy dog. " * 1800
r = c.chat.completions.create(model=M, temperature=0.0, max_tokens=30,
    messages=[{"role": "user", "content": filler + "\n\nWhat animal jumps over the dog? One word."}])
print("LONG CONTEXT:", r.choices[0].message.content, "| prompt_tokens:", r.usage.prompt_tokens)

## 6. Expose the server with a tunnel
The server only listens on `localhost` inside Kaggle, so a laptop can't reach it. `cloudflared` opens an outbound connection to Cloudflare and hands back a public `https://something.trycloudflare.com` address that forwards to port 8000. The code downloads the program once, starts it in the background and reads its log until the address appears.

Things we learned about these quick tunnels:

- The address is random and changes every time the tunnel restarts.
- There is no uptime guarantee. Once the tunnel died while the server kept running. If your laptop suddenly gets errors, check the tunnel first. Rerunning only this cell gives a new address.
- The address is public. The API key is the only thing protecting the server, so keep it secret and never commit it.

Copy the address and the key into the `.env` file on your laptop (section 11). The second cell prints the key, so clear its output before saving this notebook anywhere. The third cell checks protection from the outside: `/health` should answer 200, `/v1/models` without the key 401, and with the key 200.

In [ ]:
import re
if not os.path.exists("/kaggle/temp/cloudflared"):
    subprocess.run(["curl", "-sL", "-o", "/kaggle/temp/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
    os.chmod("/kaggle/temp/cloudflared", 0o755)
tlog = open("/kaggle/temp/tunnel.log", "w")
tunnel = subprocess.Popen(["/kaggle/temp/cloudflared", "tunnel", "--url", "http://localhost:8000",
                           "--no-autoupdate"], stdout=tlog, stderr=subprocess.STDOUT)
TUNNEL_URL = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/temp/tunnel.log").read())
    if m:
        TUNNEL_URL = m.group(0)
        print("TUNNEL URL:", TUNNEL_URL); break
else:
    print("No URL yet. Last log lines:")
    print(open("/kaggle/temp/tunnel.log").read()[-600:])

In [ ]:
print("VLLM_BASE_URL =", TUNNEL_URL)
print("VLLM_API_KEY  =", API_KEY)

In [ ]:
import urllib.error

def status(path, key=None):
    req = urllib.request.Request(TUNNEL_URL + path)
    if key:
        req.add_header("Authorization", f"Bearer {key}")
    try:
        return urllib.request.urlopen(req, timeout=20).status
    except urllib.error.HTTPError as e:
        return e.code
    except Exception as e:
        return f"error: {e}"

print("health, no key:      ", status("/health"))
print("/v1/models, no key:  ", status("/v1/models"))
print("/v1/models, with key:", status("/v1/models", API_KEY))

## 7. The benchmark data: SWE-bench Multimodal
We picked SWE-bench Multimodal because its issues include screenshots, which suits a vision model. Each row has the repo, a base commit, the issue text, the reference patch and tests, and `image_assets`, a dictionary of image links for the problem statement, the patch and the test patch.

One rule matters a lot: only the `problem_statement` images may be shown to the model. The other images come from the fix and would leak the answer. For the same reason the prompt only contains the repo at the base commit, the issue text, the screenshots, and the model's own tool outputs. It never sees the patch, the tests or any hints.

The first cell loads the dataset and lists a few rows. The second checks whether Docker exists here. It doesn't, which is why the agent and the test evaluation ran on a Mac instead, with Kaggle only serving the model.

In [ ]:
from datasets import load_dataset
import json

try:
    ds = load_dataset("SWE-bench/SWE-bench_Multimodal", split="test")
    split = "test"
except Exception as e:
    print("test split failed:", e)
    ds = load_dataset("SWE-bench/SWE-bench_Multimodal", split="dev")
    split = "dev"

print("split:", split, "| rows:", len(ds))
print("columns:", ds.column_names)

for i in range(5):
    r = ds[i]
    print(r["instance_id"], "|", r["repo"], "| text chars:", len(r["problem_statement"]))

In [ ]:
def sh(cmd): return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
print("docker binary:", sh("which docker") or "NOT FOUND")
print("docker socket:", sh("ls -l /var/run/docker.sock 2>&1"))

### A real screenshot
Here we send one real issue screenshot to the model. Large screenshots eat context, so we downscale the image to a maximum side length first and compare 1024 px against 1568 px. At 1568 px an image cost about 2K prompt tokens, which is why the benchmark capped each attempt at 2 images and downscaled to 1568 px.

The model read the text in the screenshots correctly, but it reasoned about layout and annotations shallowly and sometimes added claims that weren't in the picture. Check what it says against the image.

In [ ]:
import io
from PIL import Image

row = next(r for r in ds if r["instance_id"] == "grommet__grommet-6307")
img_url = row["image_assets"]["problem_statement"][0]      # problem_statement images only

raw = urllib.request.urlopen(urllib.request.Request(img_url, headers={"User-Agent": "Mozilla/5.0"})).read()
im = Image.open(io.BytesIO(raw))
print("original:", im.format, im.size, f"{len(raw)/1024:.0f} KB")

def to_data_url(im, max_side):
    im = im.convert("RGB")
    im.thumbnail((max_side, max_side))
    buf = io.BytesIO(); im.save(buf, "PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode(), im.size

for max_side in (1024, 1568):
    durl, size = to_data_url(im, max_side)
    r = c.chat.completions.create(model=M, temperature=0.2, max_tokens=250,
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": durl}},
            {"type": "text", "text": "This is a screenshot from a GitHub issue. Describe exactly what UI you see and anything that looks wrong or broken."}]}])
    print(f"\n--- max side {max_side} -> {size}, prompt_tokens: {r.usage.prompt_tokens}")
    print(r.choices[0].message.content)

## 8. Record GPU usage
While the benchmark ran we logged GPU use with a small script that writes one row per GPU every 5 seconds: utilization, memory, temperature, power, clocks, plus CPU and RAM. It runs in the background next to the server. In our run both GPUs were 73-88% busy, sitting at their 70 W power limit.

Kaggle reset during the run, so we saved a separate CSV for each part and downloaded them. The script opens its file in append mode, so a new name per session keeps the old data safe. The first cell writes the script, the second starts it, and the third shows the latest rows. Stop it with `monitor.terminate()` when you're done.

In [ ]:
%%writefile /kaggle/working/gpu_monitor_simple.py
import subprocess, time, csv, datetime, psutil

QUERY = "index,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw,clocks.sm,clocks.mem"
with open("/kaggle/working/gpu_monitor.csv", "a", newline="") as f:
    w = csv.writer(f)
    w.writerow(["ts_utc", "gpu", "util_pct", "mem_used_mib", "mem_total_mib", "temp_c",
                "power_w", "sm_mhz", "mem_mhz", "cpu_pct", "ram_used_gib"])
    f.flush()
    while True:
        out = subprocess.check_output(
            ["nvidia-smi", f"--query-gpu={QUERY}", "--format=csv,noheader,nounits"], text=True)
        ts = datetime.datetime.utcnow().isoformat()
        cpu = psutil.cpu_percent()
        ram = round(psutil.virtual_memory().used / 2**30, 2)
        for line in out.strip().splitlines():
            w.writerow([ts] + [x.strip() for x in line.split(",")] + [cpu, ram])
        f.flush()
        time.sleep(5)

In [ ]:
import sys
monitor = subprocess.Popen([sys.executable, "/kaggle/working/gpu_monitor_simple.py"],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("monitor PID", monitor.pid)

In [ ]:
!tail -4 /kaggle/working/gpu_monitor.csv

## 9. Load test
This measures how the server behaves when many sessions hit it at once: 1, 2, 4, 8, 16 and 32 simultaneous sessions, each sending 3 requests in a row, then one more single-session run at the end to check the server recovered. It runs on Kaggle against `localhost`, so the tunnel doesn't add noise to the numbers.

Each request asks for exactly 200 output tokens (`ignore_eos` stops the model ending early), so every request does the same amount of work. For each level the script records errors and timeouts, requests and output tokens per second, latency and time-to-first-token (p50 and p95), and whether `/health` still answers afterwards. The results are saved after every level, so a crash still leaves partial data, and the test stops early if the server stops responding.

The first cell writes the script, the second gives it the key through an environment variable, and the third runs it. Download `loadtest_results.json` from the Output panel when it finishes.

In [ ]:
%%writefile /kaggle/working/kaggle_loadtest.py
"""Concurrency load test against the local vLLM server (localhost, no tunnel)."""
import json
import os
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor

from openai import OpenAI

BASE = "http://localhost:8000"
MODEL = "ministral-3-8b"
LEVELS = [1, 2, 4, 8, 16, 32]
REQUESTS_PER_SESSION = 3
OUTPUT_TOKENS = 200
TIMEOUT_S = 120
PROMPT = "Explain what a CSS flexbox container does, in plain words."
OUT = "/kaggle/working/loadtest_results.json"


def one_request(client):
    start = time.perf_counter()
    ttft, tokens = None, 0
    try:
        stream = client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": PROMPT}], temperature=0.2,
            max_tokens=OUTPUT_TOKENS, extra_body={"ignore_eos": True},
            stream=True, stream_options={"include_usage": True})
        for chunk in stream:
            if chunk.usage:
                tokens = chunk.usage.completion_tokens
            if ttft is None and chunk.choices and chunk.choices[0].delta.content:
                ttft = time.perf_counter() - start
        return {"ok": True, "latency_s": time.perf_counter() - start, "ttft_s": ttft, "tokens": tokens}
    except Exception as e:
        kind = "timeout" if "timeout" in type(e).__name__.lower() else "error"
        return {"ok": False, "kind": kind, "error": f"{type(e).__name__}: {str(e)[:200]}",
                "latency_s": time.perf_counter() - start}


def healthy():
    try:
        return urllib.request.urlopen(BASE + "/health", timeout=10).status == 200
    except Exception:
        return False


def pct(values, p):
    values = sorted(v for v in values if v is not None)
    return round(values[min(len(values) - 1, int(round(p / 100 * (len(values) - 1))))], 3) if values else None


def run_level(client, sessions, phase):
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=sessions) as pool:
        per_session = pool.map(lambda _: [one_request(client) for _ in range(REQUESTS_PER_SESSION)], range(sessions))
        results = [r for rs in per_session for r in rs]
    wall = time.perf_counter() - start
    ok = [r for r in results if r["ok"]]
    return {
        "phase": phase, "concurrency": sessions, "requests": len(results), "ok": len(ok),
        "errors": sum(1 for r in results if not r["ok"] and r["kind"] == "error"),
        "timeouts": sum(1 for r in results if not r["ok"] and r["kind"] == "timeout"),
        "requests_per_s": round(len(ok) / wall, 3),
        "output_tokens_per_s": round(sum(r["tokens"] for r in ok) / wall, 1),
        "latency_p50_s": pct([r["latency_s"] for r in ok], 50), "latency_p95_s": pct([r["latency_s"] for r in ok], 95),
        "ttft_p50_s": pct([r["ttft_s"] for r in ok], 50), "ttft_p95_s": pct([r["ttft_s"] for r in ok], 95),
        "tokens_per_request": sorted({r["tokens"] for r in ok}),
        "healthy_after": healthy(), "wall_s": round(wall, 1),
        "sample_errors": sorted({r["error"] for r in results if not r["ok"]})[:3],
    }


def main():
    client = OpenAI(base_url=BASE + "/v1", api_key=os.environ["VLLM_API_KEY"], timeout=TIMEOUT_S, max_retries=0)
    report = []
    for sessions, phase in [(n, "load") for n in LEVELS] + [(1, "recovery")]:
        row = run_level(client, sessions, phase)
        report.append(row)
        print(f"{phase:8s} sessions={sessions:2d} ok={row['ok']}/{row['requests']} errors={row['errors']} "
              f"timeouts={row['timeouts']} req/s={row['requests_per_s']} out_tok/s={row['output_tokens_per_s']} "
              f"p50={row['latency_p50_s']}s p95={row['latency_p95_s']}s ttft_p50={row['ttft_p50_s']}s "
              f"healthy={row['healthy_after']}", flush=True)
        json.dump(report, open(OUT, "w"), indent=1)  # saved after every level
        if not row["healthy_after"]:
            print("Server unhealthy after this level, stopping.")
            break
    print("Saved", OUT)


if __name__ == "__main__":
    main()

In [ ]:
os.environ["VLLM_API_KEY"] = API_KEY

In [ ]:
!/kaggle/temp/venv/bin/python /kaggle/working/kaggle_loadtest.py

## 10. Shut down
When you're finished, stop the processes, then stop the Kaggle session with the power button at the top right. Free accounts get limited GPU hours per week. Download anything you need from `/kaggle/working` first.

While you use the endpoint, keep the Kaggle tab open and your laptop awake. Free sessions can drop without warning, and when that happens `/kaggle/temp` is wiped: rerun the install, server and tunnel cells, and update the URL and key on your laptop.

In [ ]:
for name in ("monitor", "tunnel", "server"):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        print("stopped", name)

## 11. Connect from VS Code

Everything from here runs on your laptop. The model stays on Kaggle and the laptop just sends HTTP requests, so any tool that speaks the OpenAI API can use the same URL and key.

### Set up
Install the Python and Jupyter extensions in VS Code, open a project folder, and create a virtual environment from the VS Code terminal:

```
python3 -m venv .venv
source .venv/bin/activate
pip install openai python-dotenv
```

Then pick that environment: press Cmd/Ctrl+Shift+P, choose "Python: Select Interpreter", and select the `.venv` one. If you open this notebook in VS Code, choose the same `.venv` as the kernel. Only this section will work on your laptop, since the earlier sections need the Kaggle machine.

Next, create a file called `.env` in the project folder with the two values from section 6:

```
VLLM_BASE_URL=https://your-tunnel-address.trycloudflare.com
VLLM_API_KEY=your-key
```

A few rules that save time: the URL must not end with `/` or `/v1` (a trailing slash gave us 404 errors on every request), there are no quotes or spaces around the `=`, and `.env` goes in `.gitignore` so the key never ends up on GitHub. The URL and key change whenever the server or tunnel restarts, so update `.env` and restart the notebook kernel when that happens.

### Connect and list the models
This reads `.env`, creates the client, and asks the server which models it serves. It's the fastest end-to-end test. If it prints `['ministral-3-8b']`, your laptop can reach the model.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))
BASE_URL = os.environ["VLLM_BASE_URL"].strip().rstrip("/")
client = OpenAI(base_url=BASE_URL + "/v1", api_key=os.environ["VLLM_API_KEY"].strip())
MODEL = "ministral-3-8b"

print([m.id for m in client.models.list().data])

### Basic prompt
A chat request is a list of messages, each with a role: `system` sets the behavior, `user` is what you ask. The answer comes back in `choices[0].message.content` and `usage` reports how many tokens were used. A low `temperature` like 0.2 keeps answers focused, which is better for code. `max_tokens` caps the length, and `finish_reason` is `stop` if the model finished on its own or `length` if it hit that cap.

In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise senior Python developer."},
        {"role": "user", "content": "Write a Python function that checks if a string is a palindrome."},
    ],
    temperature=0.2,
    max_tokens=300,
)
print(r.choices[0].message.content)
print("\nfinish_reason:", r.choices[0].finish_reason)
print("tokens:", r.usage.prompt_tokens, "in,", r.usage.completion_tokens, "out")

### Streaming
With `stream=True` the server sends each piece of text as it's generated instead of waiting for the full answer, which is how chat apps show text appearing live. This cell also measures time to first token, meaning how long you wait before anything shows up. From a laptop that includes the trip through the tunnel, so it will be higher than what the server itself measures. After that, our server generated roughly 19-20 tokens per second per request.

In [ ]:
import time

start = time.time()
first = None
n = 0
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain what an API is in about 100 words."}],
    max_tokens=200,
    stream=True,
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        if first is None:
            first = time.time() - start
        n += 1
        print(chunk.choices[0].delta.content, end="", flush=True)

total = time.time() - start
print(f"\n\ntime to first token: {first:.2f} s | about {n / (total - first):.0f} chunks per second after that")

### Tool calling
This is what coding agents are built on. We describe a tool to the model, and instead of answering in text it can reply with a tool call: the tool name and JSON arguments. Our code runs the tool and sends the result back as a message with role `tool`, and the model then writes the final answer. A real agent repeats this loop until the task is done.

Here the tool is faked: we pretend the file has 142 lines. If the model answers in plain text instead of calling the tool, the cell prints that. How reliably a model picks the right tool and produces valid arguments was one of the things we measured in the benchmark.

In [ ]:
import json

tools = [{"type": "function", "function": {
    "name": "get_file_line_count",
    "description": "Return how many lines a file has.",
    "parameters": {"type": "object",
                   "properties": {"path": {"type": "string"}},
                   "required": ["path"]}}}]

messages = [{"role": "user", "content": "How many lines does src/app.py have? Use the tool."}]
msg = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0).choices[0].message

if msg.tool_calls:
    call = msg.tool_calls[0]
    print("tool call:", call.function.name, call.function.arguments)
    messages.append(msg.model_dump(exclude_none=True))
    messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps({"lines": 142})})
    final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)
    print("\nanswer:", final.choices[0].message.content)
else:
    print("no tool call, the model said:", msg.content)

### Images
The model can read screenshots, which matters for bug reports. An image goes into the message `content` as an `image_url` item next to the text. It can be a public URL or a base64 data URL, which is how you send a local file. The helper below converts a file.

Each image costs roughly 1,000-3,000 tokens of the context window, so scale large screenshots down (around 1,500 pixels on the longest side is enough) and send only a few at a time. In our tests the model read text inside screenshots correctly, but its reasoning about layout and annotations was shallow and it sometimes added details that weren't in the picture, so check what it says.

In [ ]:
import base64, mimetypes

def image_to_data_url(path):
    mime = mimetypes.guess_type(path)[0] or "image/png"
    with open(path, "rb") as f:
        return f"data:{mime};base64," + base64.b64encode(f.read()).decode()

# a public URL...
image = {"type": "image_url", "image_url": {
    "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"}}
# ...or a local file (uncomment and change the path)
# image = {"type": "image_url", "image_url": {"url": image_to_data_url("screenshot.png")}}

r = client.chat.completions.create(
    model=MODEL, max_tokens=200,
    messages=[{"role": "user", "content": [image, {"type": "text", "text": "Describe what you see."}]}],
)
print(r.choices[0].message.content)
print("\nprompt_tokens:", r.usage.prompt_tokens)

## What we did with this setup
We used this endpoint to benchmark the model on SWE-bench Multimodal, which is real GitHub issues with screenshots. A small agent on the laptop gave the model tools to list files, read files, search, edit and finish, and ran 10 issues three times each (30 attempts). The model only saw the repo at the base commit, the issue text and screenshots, and its own tool outputs, never the reference patch or tests. Results from the load test and the Docker evaluation of the patches will go here once they're done.

## Troubleshooting
The errors we actually hit, and what fixed them.

- **Images fail with "Expected there to be N prompt placeholders ... found 0":** the model was loaded in Mistral's own format. Start the server with `--tokenizer-mode hf --config-format hf --load-format safetensors` (section 3).
- **404 on every request:** a trailing slash or an extra `/v1` in `VLLM_BASE_URL`. The code adds `/v1` itself.
- **401:** the key doesn't match the running server. Every server launch generates a new key.
- **530 / "can't resolve host":** the tunnel is down or the Kaggle session ended. Rerun the tunnel cell (new URL), or start again from section 2 if Kaggle was reset.
- **400 about context length:** the prompt plus images is longer than `--max-model-len`. Send fewer or smaller images.
- **Server crashes at startup with a Triton "shared memory" error:** the model's attention kernel needs more shared memory than a T4 has. We hit this with Gemma 4 E4B. Pick a model that runs on compute capability 7.5, so no FP8, BF16 or FlashAttention 2.
- **Server exits with no clear error:** read the end of the log, and search it for `Error` or `Traceback`. Out-of-memory problems show up there too.

## Also part of the project
- **Evaluating patches:** the benchmark's agent and the Docker-based test runs happened on a Mac, because Kaggle has no Docker. Kaggle only hosted the model.